<a href="https://colab.research.google.com/github/Venu-max/NASSCOM-AI/blob/main/Day4_U8_%E2%80%94_Data_Cleaning_%26_Preprocessing_(Part_2)_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)



Setup complete. pandas 2.2.2


In [3]:
# -----------------------------------------------------------
# A SMALL, ALREADY-CLEAN DATASET (Part 1 did the cleaning)
# -----------------------------------------------------------
# Mixed columns: an unordered category (city), an ordered category
# (size), two numeric features on very different scales, and a target.
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})


In [ ]:
##1. Encoding categorical data

In [4]:
# -----------------------------------------------------------
# 🔹 1A. ONE-HOT ENCODING (for UNORDERED categories)
# -----------------------------------------------------------

# 'city' has no natural order -> one 0/1 column per category
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe

,city_delhi,city_mumbai,city_pune
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0
6,0,0,1
7,1,0,0


In [7]:
# -----------------------------------------------------------
# 🔹 1B. LABEL / ORDINAL ENCODING (for ORDERED categories)
# -----------------------------------------------------------

# 'size' has a real order: small < medium < large -> map to 0,1,2
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]

,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


In [ ]:
##LAB EXERCISE 1 — Encode the columns
#One-hot encode city again, but this time use pd.get_dummies on the whole DataFrame (pass columns=['city']).
#Ordinal-encode size using size_order (you can reuse size_code).
#In a comment, explain why one-hot is wrong for size and ordinal is wrong for city.

In [6]:
# 1. One-hot encode only the 'city' column of the whole DataFrame
# pd.get_dummies() creates separate binary (0/1) columns for each city.
# The original 'city' column is removed and replaced with new columns.

df_encoded = pd.get_dummies(df, columns=['city'], prefix='city').astype({
    'city_delhi': int,
    'city_mumbai': int,
    'city_pune': int
})

# Display the encoded DataFrame
print(df_encoded)


# 2. Ordinal-encode the 'size' column using the size_order dictionary
# map() replaces 'small', 'medium', and 'large' with 0, 1, and 2 respectively.

size_order = {'small': 0, 'medium': 1, 'large': 2}

df_encoded['size_code'] = df_encoded['size'].map(size_order)

# Display the size and its encoded values
print(df_encoded[['size', 'size_code']])


# 3. Explanation:
# One-hot encoding is NOT suitable for 'size' because 'size' has a natural order
# (small < medium < large). One-hot encoding loses this ordering information.
#
# Ordinal encoding is NOT suitable for 'city' because cities have no natural order.
# Assigning numbers like Pune=0, Delhi=1, Mumbai=2 incorrectly suggests an order
# or ranking that does not exist.

     size  age  income  bought  size_code  city_delhi  city_mumbai  city_pune
0   small   25   38000       0          0           0            0          1
1   large   41   92000       1          2           1            0          0
2  medium   33   55000       0          1           0            1          0
3  medium   29   47000       0          1           0            0          1
4   small   52  120000       1          0           1            0          0
5   large   38   76000       1          2           0            1          0
6   large   46   88000       1          2           0            0          1
7   small   22   41000       0          0           1            0          0
     size  size_code
0   small          0
1   large          2
2  medium          1
3  medium          1
4   small          0
5   large          2
6   large          2
7   small          0


In [ ]:
##2. Feature scaling

In [9]:
# -----------------------------------------------------------
# 🔹 2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES
# -----------------------------------------------------------

# income (tens of thousands) dwarfs age (tens). A distance-based
# model would treat income as nearly all that matters.
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])

        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0


In [8]:
# -----------------------------------------------------------
# 🔹 2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)        # mean 0, std 1
m = MinMaxScaler().fit_transform(num)          # range [0, 1]

print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))

Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


In [ ]:
##LAB EXERCISE 2 — Scale a feature
#Using the income column:
#Standardise it with StandardScaler and confirm the result's mean is ~0 and std ~1.
#Min-Max scale it and confirm the result's min is 0 and max is 1.
#In a comment, name one model type that needs scaling and one that doesn't.

In [10]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Select the income column (2D DataFrame required by sklearn)
income = df[['income']]

# -------------------------------------------------------
# 1. Standardise the income column
# StandardScaler transforms the data so that:
# Mean ≈ 0 and Standard Deviation ≈ 1
# -------------------------------------------------------

scaler = StandardScaler()

income_std = scaler.fit_transform(income)

# Check the mean and standard deviation
print("Mean after Standardisation:", income_std.mean())
print("Standard Deviation after Standardisation:", income_std.std())

# -------------------------------------------------------
# 2. Min-Max Scale the income column
# MinMaxScaler transforms the data into the range [0,1]
# -------------------------------------------------------

minmax = MinMaxScaler()

income_minmax = minmax.fit_transform(income)

# Check the minimum and maximum values
print("Minimum after Min-Max Scaling:", income_minmax.min())
print("Maximum after Min-Max Scaling:", income_minmax.max())

# -------------------------------------------------------
# 3. Comments
# Needs scaling: K-Nearest Neighbors (KNN), Support Vector Machine (SVM), K-Means
# Doesn't need scaling: Decision Tree, Random Forest
# -------------------------------------------------------

Mean after Standardisation: -2.7755575615628914e-17
Standard Deviation after Standardisation: 1.0
Minimum after Min-Max Scaling: 0.0
Maximum after Min-Max Scaling: 1.0000000000000002


In [ ]:
##3. Avoiding data leakage
#Golden rule: fit scalers/encoders on the training set only, then apply to the test set. Fitting on all the data lets test information leak in.

In [11]:

# -----------------------------------------------------------
# 🔹 3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT + transform on train
X_test_s  = scaler.transform(X_test)        # only TRANSFORM on test
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))

train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


In [ ]:
##LAB EXERCISE 3 — Spot the leak
#The two lines below show the wrong way (fit on all data) and the right way (fit on train). Run them and compare the learned means.
#In a comment, explain why fitting on all the data before splitting is a problem.

In [12]:
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------
# 1a. WRONG: Fit on ALL data before splitting
# This learns the mean and standard deviation from the
# entire dataset, including the test data (data leakage).
# -------------------------------------------------------

wrong_mean = StandardScaler().fit(X).mean_

# -------------------------------------------------------
# 1b. RIGHT: Fit only on the TRAIN data
# This learns the statistics only from the training set.
# -------------------------------------------------------

right_mean = StandardScaler().fit(X_train).mean_

# Print the learned means
print("fit-on-all mean  :", wrong_mean.round(1))
print("fit-on-train mean:", right_mean.round(1))

# -------------------------------------------------------
# 2. Why fitting on all data is a problem:
#
# Fitting the scaler on the entire dataset uses information
# from the test data (such as its mean and standard deviation).
# This causes data leakage because the model indirectly learns
# about the test set before evaluation. As a result, the test
# accuracy may appear better than it really is and will not
# reflect performance on truly unseen data.
# -------------------------------------------------------

fit-on-all mean  : [3.5800e+01 6.9625e+04]
fit-on-train mean: [3.45e+01 6.90e+04]


In [ ]:
##4. Feature engineering

In [13]:
# -----------------------------------------------------------
# 🔹 4A. COMBINE & BIN EXISTING COLUMNS
# -----------------------------------------------------------

fe = df.copy()
# combine: income per year of age (a crude 'earning rate')
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)
# bin: turn continuous age into life-stage buckets
fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]

,age,income,income_per_age,age_group
0,25,38000,1520.0,young
1,41,92000,2244.0,mid
2,33,55000,1667.0,mid
3,29,47000,1621.0,young
4,52,120000,2308.0,senior
5,38,76000,2000.0,mid
6,46,88000,1913.0,senior
7,22,41000,1864.0,young


In [14]:
# -----------------------------------------------------------
# 🔹 4B. EXTRACT FROM A DATE
# -----------------------------------------------------------

dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()   # Monday, Tuesday, ...
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d

,date,day_of_week,month,is_weekend
0,2024-01-06,Saturday,1,True
1,2024-03-15,Friday,3,False
2,2024-07-21,Sunday,7,True
3,2024-12-25,Wednesday,12,False


In [ ]:
##LAB EXERCISE 4 — Engineer two features
#On a fresh copy ex = df.copy():
#Create high_earner = True when income is above its median.
#Bin income into 3 labelled buckets with pd.cut (e.g. low / mid / high).
#Show the new columns next to income.

In [17]:
# Create a copy of the original DataFrame
ex = df.copy()

# -------------------------------------------------------
# 1. Create a new feature 'high_earner'
# If income is greater than the median income,
# the value will be True, otherwise False.
# -------------------------------------------------------

ex['high_earner'] = ex['income'] > ex['income'].median()

# -------------------------------------------------------
# 2. Bin the income column into 3 categories
# using pd.cut()
# low    -> low income
# mid    -> medium income
# high   -> high income
# -------------------------------------------------------

ex['income_group'] = pd.cut(
    ex['income'],
    bins=3,
    labels=['low', 'mid', 'high']
)

# -------------------------------------------------------
# 3. Display the income column along with
# the newly created features
# -------------------------------------------------------

print(ex[['income', 'high_earner', 'income_group']])

   income  high_earner income_group
0   38000        False          low
1   92000         True          mid
2   55000        False          low
3   47000        False          low
4  120000         True         high
5   76000         True          mid
6   88000         True          mid
7   41000        False          low


In [ ]:
##5. The preprocessing pipeline

In [15]:
# -----------------------------------------------------------
# 🔹 5A. ONE LEAK-FREE PIPELINE: PREPROCESS + MODEL
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']

# scale the numbers, one-hot the categories — all in one object
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])


In [16]:
# -----------------------------------------------------------
# 🔹 5B. FIT THE WHOLE THING IN ONE CALL
# -----------------------------------------------------------

X = df[num_cols + cat_cols]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

pipe.fit(X_train, y_train)        # preprocessing fitted on TRAIN only — no leakage
acc = pipe.score(X_test, y_test)  # transforms test with train-fitted steps
print('Test accuracy:', round(acc, 2))
print('(small toy dataset — the point is the leak-free workflow, not the score)')

Test accuracy: 1.0
(small toy dataset — the point is the leak-free workflow, not the score)


In [ ]:
##LAB EXERCISE 5 — Build your own pipeline
#Build a Pipeline of just MinMaxScaler + LogisticRegression on the numeric columns (age, income).
#Split the data, then fit on the training set.
#Print the test accuracy with .score(...).

In [18]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Select numeric features and target variable
Xn = df[['age', 'income']]
yn = df['bought']

# -------------------------------------------------------
# 1. Create a Pipeline
# Step 1: MinMaxScaler scales the features to the range [0,1]
# Step 2: LogisticRegression trains the classification model
# -------------------------------------------------------

pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', LogisticRegression())
])

# -------------------------------------------------------
# 2. Split the dataset into training and testing sets
# 75% for training and 25% for testing
# random_state=0 ensures the same split every time
# -------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    Xn,
    yn,
    test_size=0.25,
    random_state=0
)

# Train the pipeline
pipe.fit(X_train, y_train)

# -------------------------------------------------------
# 3. Calculate and print the test accuracy
# score() predicts the test data and computes accuracy
# -------------------------------------------------------

accuracy = pipe.score(X_test, y_test)

print("Test Accuracy:", round(accuracy, 2))

Test Accuracy: 1.0
